**CancerGenomicEpidemiology - Mutational Signature Analysis Demo**

In this demo you will learn to use some of the available tools to perform a simple mutational signature analysis. You will see examples of signatures which are caused by environmental exposures and learn how to integrate basic metadata to interpret results.

For this excercise, please work in small groups. You can all run the exercise individualy or norminate someone to run it. After completing steps 3, 4 and 5, stop and review the outputs and discuss the questions with your groups before moving on.

**Step 1** - Install and import the packages needed for the analysis

In [ ]:
!pip install sigprofilerassignment pandas matplotlib

In [ ]:
# Import libraries
from SigProfilerAssignment import Analyzer as Analyze
import sigProfilerPlotting as sigPlt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os


**Step 2** - Import a synthetic dataset which has three different cancer types to practise mutational signature analysis.

Note - this is a very simplistic dataset, in real datasets there is typically more variation and additional signatures present


Before we start analysing, we will check the format of the import matrix. We expect this to be the number of contexts (e.g 96 for SBS96 contexts) x the number of samples (50 in this case)


In [ ]:
!git clone https://github.com/WCSCourses/Cancer_Genomic_Epidemiology_2026.git

In [ ]:
# Load SBS96 matrix
sbs96 = pd.read_csv("/content/Cancer_Genomic_Epidemiology_2026/course_data_2026/Mutational_Signatures_Data/SBS96.synthetic.tsv", sep="\t", index_col=0)
sbs96.head()

In [ ]:
print("Matrix shape (contexts x samples):", sbs96.shape)

**Step 3** - Make mutational spectra for each synthetic sample using SigProfilerPlotting. Spectra are a composite of all the mutational signatures active in a given sample. You can find more information about SigProfilerPlotting at https://github.com/AlexandrovLab/SigProfilerPlotting.

In [ ]:
os.makedirs("Plots_SBS96", exist_ok=True)

sigPlt.plotSBS(
    matrix_path=sbs96,
    output_path="./Plots_SBS96",
    project="Synthetic_Samples",
    plot_type="96"
)


**Step 3 Questions**

Open the SBS96 spectra plots for the synthetic samples:

*   Which samples show strong C>T peaks?
*   Which samples show broad C>A enrichment?
*   Which samples have do not any particulary prominent peaks (sometimes referred to as flat spectra)




**Step 4** - COSMIC reference signature to the synthetic dataset using SigProfilerAssignment. More information on how to run SigProfilerAssignment can be found at https://github.com/AlexandrovLab/SigProfilerAssignment, and detailed information on the outputs can be found at https://osf.io/mz79v/wiki?wiki=pzvn6


In [ ]:
from SigProfilerAssignment import Analyzer as Analyze

os.makedirs("Assignment_Synthetic", exist_ok=True)

Analyze.cosmic_fit(
    input_type="matrix",                          # Our input is a mutation count matrix
    samples=sbs96,                                # The sbs96 DataFrame
    output="Assignment_Synthetic",                # Output folder
    genome_build="GRCh38",                        # Reference Genome
    context_type="96",                            # SBS96 contexts
    cosmic_version=3.4,                           # COSMIC version
    collapse_to_SBS96=True
)

**Step 4 Questions:**


Using the outputs from SigProfilerAssignment (Assignment_Solution_Activities.txt, Assignment_Solution_Activity_Plots.pdf, Assignment_Solution_TMB_plot.pdf):

*   How many COSMIC reference signatures are present in the synthetic dataset?
*   Which signatures are found in all sample types and which are restricted. What does that suggest about the biological mechanisms?










**Step 5** - Visualise the Results

We will make some plots comparing the signatures in the three cancer types and then look at the subgroups of breast and lung cancers


In [ ]:
# Merge signature activities with metadata
activities_path = "/content/Assignment_Synthetic/Assignment_Solution/Activities/Assignment_Solution_Activities.txt"

activities = pd.read_table(activities_path, sep="\t", index_col=0)
activities = activities.loc[:, ~activities.columns.str.contains('^Unnamed')]  # drop extra columns

metadata = pd.read_csv("/content/Cancer_Genomic_Epidemiology_2026/course_data_2026/Mutational_Signatures_Data/sample_metadata.tsv", sep="\t").set_index("Sample_ID")
activities = activities.loc[metadata.index]  # ensure sample order matches
data = metadata.join(activities)


In [ ]:
signature_cols = activities.columns
grouped = data.groupby("Cancer_Type")[signature_cols].mean()

import matplotlib.pyplot as plt

# Ensure inline display
%matplotlib inline

# Filter to signatures present in at least one cancer type
present_sigs = grouped.columns[(grouped.sum(axis=0) > 0)]
grouped_present = grouped[present_sigs]

# Create figure
fig, ax = plt.subplots(figsize=(10,6))

# Plot only present signatures
grouped_present.plot(
    kind='bar',
    stacked=True,
    colormap='tab20',
    ax=ax
)

# Labels and title
ax.set_ylabel("Mean Number of Mutations")
ax.set_xlabel("Cancer Type")
ax.set_title("Mean Signature Activity per Cancer Type")

# Layout and save
plt.tight_layout()
fig.savefig("/content/Mean_Signature_Activity_per_Cancer_Type.png", dpi=300, bbox_inches='tight')

# Show inline
plt.show()



In [ ]:
# Subset Lung samples
lung_data = data[data["Cancer_Type"] == "Lung"]

# Group by Subgroup (Smoker vs NonSmoker)
lung_grouped = lung_data.groupby("Subgroup")[signature_cols].mean()

# Filter to signatures present in at least one subgroup
present_sigs_lung = lung_grouped.columns[(lung_grouped.sum(axis=0) > 0)]
lung_grouped_present = lung_grouped[present_sigs_lung]

# Plot
fig, ax = plt.subplots(figsize=(8,5))
lung_grouped_present.plot(kind='bar', stacked=True, colormap='tab20', ax=ax)
ax.set_ylabel("Mean Number of Mutations")
ax.set_xlabel("Lung Subgroup")
ax.set_title("Mean Signature Activity: Lung")
plt.xticks(rotation=0)
plt.tight_layout()
fig.savefig("/content/Mean_Signature_Activity_Lung_Subgroups.png", dpi=300, bbox_inches='tight')
plt.show()



In [ ]:
# Subset Breast samples
breast_data = data[data["Cancer_Type"] == "Breast"]

# Group by Subgroup
breast_grouped = breast_data.groupby("Subgroup")[signature_cols].mean()

# Filter to signatures present in at least one subgroup
present_sigs_breast = breast_grouped.columns[(breast_grouped.sum(axis=0) > 0)]
breast_grouped_present = breast_grouped[present_sigs_breast]

# Plot
fig, ax = plt.subplots(figsize=(8,5))
breast_grouped_present.plot(kind='bar', stacked=True, colormap='tab20', ax=ax)
ax.set_ylabel("Mean Number of Mutations")
ax.set_xlabel("Breast Subgroup")
ax.set_title("Mean Signature Activity: Breast")
plt.xticks(rotation=0)
plt.tight_layout()
fig.savefig("/content/Mean_Signature_Activity_Breast_Subgroups.png", dpi=300, bbox_inches='tight')
plt.show()

**Step 5 Questions:**


Using the outputs from SigProfilerAssignment and the generated plots:

*   Identify the dominant signature(s) for each subgroup and find the etiology using the COSMIC website https://cancer.sanger.ac.uk/signatures/sbs/
*   What additional metadata could you collect from patients to investigate variation in the levels of SBS4 in Lung cancers?
*  For signatures which contribute small numbers of mutations, how easy is it to spot the signatures in the mutational spectra? What other factors might make it harder to see a signature in the mutational spectra?





